# SOEN Disruption Prediction — PCA-8 seq2seq Binary Classification

Train a neuromorphic SOEN model on the PCA-8 + 100× decimated + flattop dataset for **per-timestep binary disruption classification**, directly comparable to the TCN baseline.

**Task**: `seq2seq` with `cross_entropy` loss — predict `{0=clear, 1=disruptive}` at every timestep.

**Data**: Same `all_data.h5` from `preprocessing_pca8_100x_flattop.ipynb` → `(N, 8, 7812)`.

**Model**: SOEN `8IN→28H(rec)→2ClassReadout` with hardware constraints:
- On-chip connections (J_0_to_1, J_1_to_1): trainable, 3-bit QAT, bounds [-0.14, 0.14]
- Physics params (phi_offset, bias_current, gamma): **fixed** per hardware spec
- Hidden→readout (J_1_to_2): **fixed** one-to-one coupling J=0.5

**Deliverable**: Quantized `.soen` checkpoint for hardware deployment.

In [ ]:
import sys
import subprocess
from pathlib import Path

import numpy as np
import h5py
import yaml

# ── Find soen_toolkit src ─────────────────────────────────────────
# The soenhardware/soen-toolkit uses `X | None` type syntax throughout,
# which fails at runtime on this Python. We locate the source but do NOT
# import it in-process. All soen_toolkit operations (model build, training,
# audit) run via subprocess where the correct Python/env handles it.
_SOEN_SRC_CANDIDATES = [
    Path("/home/idies/workspace/Temporary/dpark1/scratch/soenhardware/soen-toolkit/src"),
    Path("/home/idies/workspace/Temporary/dpark1/scratch/SOEN/soenre2/src"),
    Path("/Users/davidpark/Documents/Cursor/soenhardware/soen-toolkit/src"),
]

SOEN_SRC = None
for _c in _SOEN_SRC_CANDIDATES:
    if (_c / "soen_toolkit" / "__init__.py").exists():
        SOEN_SRC = _c
        break
if SOEN_SRC is None:
    raise FileNotFoundError("soen_toolkit src not found")

TUTORIAL_DIR = SOEN_SRC / "soen_toolkit" / "tutorial_notebooks" / "time_to_event_tutorial"
print(f"soen_toolkit src: {SOEN_SRC}")
print(f"tutorial_dir:     {TUTORIAL_DIR}")

# ── Helper: run Python code in a subprocess with soen_toolkit available ──
def run_soen_python(code: str, check: bool = True) -> subprocess.CompletedProcess:
    """Run Python code in subprocess with PYTHONPATH set to soen_toolkit src."""
    env = {**__import__('os').environ, "PYTHONPATH": str(SOEN_SRC)}
    return subprocess.run(
        [sys.executable, "-c", code],
        env=env, check=check, capture_output=True, text=True,
    )

# Verify soen_toolkit is importable in subprocess
_r = run_soen_python("import soen_toolkit; print(soen_toolkit.__file__)")
print(f"subprocess check: {_r.stdout.strip()}")

# ── Paths ─────────────────────────────────────────────────────────
PCA8_H5 = Path("/home/idies/workspace/Storage/yhuang2/persistent/ecei_mc/pca8_100x_flattop/all_data.h5")

SOEN_DIR = Path("soen_training")
DATASET_DIR = SOEN_DIR / "datasets"
MODEL_DIR = SOEN_DIR / "model_specs"
CONFIG_DIR = SOEN_DIR / "training_configs"
RESULTS_DIR = SOEN_DIR / "results"

for d in [DATASET_DIR, MODEL_DIR, CONFIG_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"PCA8 source:      {PCA8_H5}")
print(f"SOEN dir:         {SOEN_DIR.resolve()}")

## 1. Convert PCA8 H5 → SOEN HDF5 format

SOEN expects `(N, T, D)` layout with `labels` as int64 for classification.

| PCA8 H5 | SOEN H5 |
|---------|---------|
| `X` (N, 8, 7812) float32 | `data` (N, 7812, 8) float32 |
| `target` (N, 7812) float32 {0,1} | `labels` (N, 7812) int64 {0,1} |
| `weight` (N, 7812) float32 {0,1} | `target_mask` (N, 7812) bool |
| — | `input_mask` (N, 7812) bool (all True) |

In [ ]:
SOEN_H5 = DATASET_DIR / "pca8_disruption_seq2seq_2class.h5"

with h5py.File(PCA8_H5, "r") as src, h5py.File(SOEN_H5, "w") as dst:
    for split in ("train", "val", "test"):
        g = dst.create_group(split)

        # X: (N, 8, T) → (N, T, 8) — SOEN expects channels-last
        X = np.asarray(src[f"{split}/X"])          # (N, 8, 7812)
        X = np.transpose(X, (0, 2, 1))             # (N, 7812, 8)
        g.create_dataset("data", data=X, dtype=np.float32)

        # target: float32 {0.0, 1.0} → int64 {0, 1}
        target = np.asarray(src[f"{split}/target"])  # (N, 7812)
        labels = target.astype(np.int64)
        g.create_dataset("labels", data=labels, dtype=np.int64)

        # weight → target_mask: where loss should be computed
        weight = np.asarray(src[f"{split}/weight"])  # (N, 7812)
        target_mask = (weight > 0).astype(bool)
        g.create_dataset("target_mask", data=target_mask)

        # input_mask: all timesteps are valid (pre-subsequenced)
        input_mask = np.ones(X.shape[:2], dtype=bool)
        g.create_dataset("input_mask", data=input_mask)

        N, T, D = X.shape
        n_pos = int(labels.sum())
        n_valid = int(target_mask.sum())
        print(f"  {split}: N={N}, T={T}, D={D}, "
              f"pos_labels={n_pos}/{n_valid} valid ({n_pos/max(n_valid,1)*100:.1f}%)")

print(f"\nSaved: {SOEN_H5}")

## 2. Build SOEN model

Architecture: `Linear(8) → SingleDendrite(28, recurrent) → DendriteReadout(2)`

**Fixed** (hardware spec): phi_offset=0.23, bias_current=1.7, gamma_plus/minus, J_1_to_2=0.5
**Trainable** (3-bit QAT): J_0_to_1 (input→hidden), J_1_to_1 (recurrent)

In [ ]:
MODEL_PATH = MODEL_DIR / "8IN_28H_2ClassSeq2Seq.soen"

# Build model via subprocess (soen_toolkit uses X|None syntax that fails in-process)
_build_code = f"""
import sys
sys.path.insert(0, {str(TUTORIAL_DIR)!r})
from notebook_utils import build_direct_readout_model
from pathlib import Path

build_direct_readout_model(
    model_path=Path({str(MODEL_PATH.resolve())!r}),
    output_dim=2,
    input_dim=8,
    hidden_dim=28,
    dt_ns=10.0,
    phi_offset=0.23,
    phi_offset_learnable=False,
    bias_current=1.7,
    bias_current_learnable=False,
    gamma_plus=2.3508e-5,
    gamma_minus=2.6995e-5,
    gamma_minus_learnable=False,
)
print("OK")
"""
_r = run_soen_python(_build_code)
if _r.returncode != 0:
    print("STDERR:", _r.stderr)
    raise RuntimeError("Model build failed")
print(f"Model saved: {MODEL_PATH}")
print(f"  Input:  8 (PCA components)")
print(f"  Hidden: 28 (SingleDendrite, recurrent)")
print(f"  Output: 2 (binary per-timestep classification)")
print(f"  File exists: {MODEL_PATH.exists()}")

## 3. Build training config

Write YAML matching the SOEN tutorial convention, but with:
- `mapping: seq2seq` + `losses: cross_entropy` → per-timestep binary classification
- `num_classes: 2`
- No `time_pooling` (seq2seq outputs at every timestep)
- QAT on J_0_to_1 and J_1_to_1 only (3-bit, [-0.14, 0.14])
- Fixed-length data (all subsequences are 7812)

In [ ]:
EXPERIMENT_NAME = "pca8_disruption_seq2seq_2class"
SEQ_LEN = 7812
DT_NS = 10.0
MAX_EPOCHS = 200
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
BACKEND = "jax"  # or "torch"

CONFIG_PATH = CONFIG_DIR / f"training_config_{EXPERIMENT_NAME}.yaml"

# ── Load the tutorial base YAML and override for our task ──
BASE_YAML = TUTORIAL_DIR / "training" / "training_configs" / "bnl_tutorial_base.yaml"
cfg = yaml.safe_load(BASE_YAML.read_text())

# ── Training: seq2seq + cross_entropy ──
cfg["training"]["mapping"] = "seq2seq"
cfg["training"]["losses"] = [{"name": "cross_entropy", "weight": 1.0}]
cfg["training"]["max_epochs"] = MAX_EPOCHS
cfg["training"]["batch_size"] = BATCH_SIZE
cfg["training"]["optimizer"]["lr"] = LEARNING_RATE
cfg["training"]["checkpoint_every_n_epochs"] = 10
cfg["training"]["run_test_after_fit"] = False

# ── Data ──
cfg["data"]["data_path"] = str(SOEN_H5.resolve())
cfg["data"]["variable_length"] = False
cfg["data"]["target_seq_len"] = SEQ_LEN
cfg["data"]["sequence_length"] = SEQ_LEN
cfg["data"]["total_time_ns"] = float(SEQ_LEN) * DT_NS  # 78120.0
cfg["data"]["num_classes"] = 2
cfg["data"]["pad_value"] = 0.0
cfg["data"]["min_scale"] = 0.0
cfg["data"]["max_scale"] = 1.0

# ── Model: no time_pooling for seq2seq ──
cfg["model"]["base_model_path"] = str(MODEL_PATH.resolve())
cfg["model"]["backend"] = BACKEND
cfg["model"].pop("time_pooling", None)

# ── Logging ──
cfg["logging"]["project_dir"] = str(RESULTS_DIR.resolve())
cfg["logging"]["experiment_name"] = EXPERIMENT_NAME

# ── Callbacks ──
cfg["callbacks"]["lr_scheduler"]["max_lr"] = LEARNING_RATE
cfg["callbacks"]["stateful_training"]["enable_for_validation"] = False

with CONFIG_PATH.open("w") as f:
    yaml.safe_dump(cfg, f, default_flow_style=False, sort_keys=False)

print(f"Config saved: {CONFIG_PATH}")
print(f"  Based on: {BASE_YAML}")
print(f"  mapping: seq2seq + cross_entropy (per-timestep binary)")
print(f"  seq_len: {SEQ_LEN}, dt_ns: {DT_NS}, num_classes: 2")
print(f"  QAT: 3-bit on J_0_to_1, J_1_to_1 (bounds [-0.14, 0.14])")
print(f"  epochs: {MAX_EPOCHS}, batch: {BATCH_SIZE}, lr: {LEARNING_RATE}")

## 4. Train

Invokes `soen_toolkit.training` via subprocess (same as tutorial `02_train_models.ipynb`).
Saves `initial.soen` and `last.soen` checkpoints in `.soen` format.

In [ ]:
# Training via subprocess (same as notebook_utils.run_training)
print(f"Launching SOEN training: {CONFIG_PATH}")
print(f"  This may take a while for {MAX_EPOCHS} epochs on seq_len={SEQ_LEN}...")

_train_result = subprocess.run(
    [sys.executable, "-m", "soen_toolkit.training", str(CONFIG_PATH.resolve())],
    env={**__import__('os').environ, "PYTHONPATH": str(SOEN_SRC)},
)
if _train_result.returncode != 0:
    raise RuntimeError(f"Training failed with exit code {_train_result.returncode}")
print("Training complete.")

## 5. Plot loss curves

In [ ]:
import matplotlib.pyplot as plt
import json as _json

# Read TensorBoard scalars via subprocess
_tb_code = f"""
import sys, json
sys.path.insert(0, {str(TUTORIAL_DIR)!r})
from notebook_utils import read_training_scalars_with_fallback
scalars = read_training_scalars_with_fallback({str(CONFIG_PATH.resolve())!r})
out = {{}}
for tag, df in scalars.items():
    out[tag] = {{"step": df["step"].tolist(), "value": df["value"].tolist()}}
print(json.dumps(out))
"""
_r = run_soen_python(_tb_code, check=False)
scalars = {}
if _r.returncode == 0 and _r.stdout.strip():
    try:
        raw = _json.loads(_r.stdout.strip().split("\n")[-1])
        scalars = {k: v for k, v in raw.items()}
    except _json.JSONDecodeError:
        print("Warning: could not parse TensorBoard output")
else:
    print(f"Warning: TensorBoard read failed. stderr: {_r.stderr[:500] if _r.stderr else 'none'}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for tag, data in scalars.items():
    if "loss" in tag.lower() and "epoch" in tag.lower():
        label = "train" if "train" in tag.lower() else "val"
        ax.plot(data["step"], data["value"], label=label)
ax.set_xlabel("Epoch")
ax.set_ylabel("Cross-Entropy Loss")
ax.set_title("Training Loss")
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
found_acc = False
for tag, data in scalars.items():
    if "acc" in tag.lower() and "epoch" in tag.lower():
        label = "train" if "train" in tag.lower() else "val"
        ax.plot(data["step"], data["value"], label=label)
        found_acc = True
if found_acc:
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Accuracy")
    ax.set_title("Per-timestep Accuracy")
    ax.legend()
    ax.grid(True, alpha=0.3)
else:
    ax.text(0.5, 0.5, "No accuracy metrics logged", ha="center", va="center", transform=ax.transAxes)

plt.tight_layout()
plt.show()

## 6. Audit trained weights and quantize to 3-bit `.soen`

Verify:
1. All trainable connections (J_0_to_1, J_1_to_1) are within [-0.14, 0.14]
2. Fixed parameters (phi_offset, bias_current, gamma, J_1_to_2) are unchanged
3. QAT produced valid 3-bit (9-level) quantized weights

**Output**: `last_quant_3bit9lvl.soen` — the hardware deliverable.

In [ ]:
import json as _json

# Audit and quantize via subprocess
_audit_code = f"""
import sys, json
sys.path.insert(0, {str(TUTORIAL_DIR)!r})
from pathlib import Path
from notebook_utils import audit_and_quantize_latest_run

audit = audit_and_quantize_latest_run(
    results_dir=Path({str(RESULTS_DIR.resolve())!r}),
    config_path=Path({str(CONFIG_PATH.resolve())!r}),
    experiment_name={EXPERIMENT_NAME!r},
    target_connections=["J_0_to_1", "J_1_to_1"],
    weight_min=-0.14,
    weight_max=0.14,
    quant_levels=9,
)

# Serialize for transfer back to notebook
out = {{
    "bounds_ok": audit["bounds_ok"],
    "fixed_params_unchanged": audit["fixed_params_unchanged"],
    "qat_active_in_config": audit["qat_active_in_config"],
    "bounds_stats": {{k: {{kk: float(vv) if isinstance(vv, (int, float)) else vv for kk, vv in v.items()}} for k, v in audit["bounds_stats"].items()}},
    "quant_levels_present": {{k: int(v) for k, v in audit.get("quant_levels_present", {{}}).items()}},
    "checkpoint_dir": str(audit["checkpoint_dir"]),
    "quantized_checkpoint": str(audit["quantized_checkpoint"]),
}}
loss_info = audit.get("loss_info", {{}})
out["loss_info"] = {{k: (float(v) if isinstance(v, (int, float)) else str(v)) for k, v in loss_info.items()}}
print(json.dumps(out))
"""

_r = run_soen_python(_audit_code)
if _r.returncode != 0:
    print("STDERR:", _r.stderr)
    raise RuntimeError("Audit failed")

audit = _json.loads(_r.stdout.strip().split("\n")[-1])

print("=== Audit Results ===")
print(f"  Bounds OK:              {audit['bounds_ok']}")
print(f"  Fixed params unchanged: {audit['fixed_params_unchanged']}")
print(f"  QAT active in config:   {audit['qat_active_in_config']}")

print("\n=== Weight Bounds ===")
for conn, stats in audit["bounds_stats"].items():
    print(f"  {conn}: min={stats['min']:.6f}, max={stats['max']:.6f}, in_bounds={stats['in_bounds']}")

print("\n=== Quantization Levels ===")
for conn, n_lvl in audit.get("quant_levels_present", {}).items():
    print(f"  {conn}: {n_lvl} unique quantized values")

print("\n=== Loss (float vs quantized) ===")
loss_info = audit.get("loss_info", {})
print(f"  Train loss (float32):   {loss_info.get('train_loss_float', 'N/A')}")
print(f"  Train loss (quantized): {loss_info.get('train_loss_quantized', 'N/A')}")

print(f"\n=== Hardware Deliverable ===")
print(f"  {audit['quantized_checkpoint']}")

## 7. Verify checkpoint structure

Load the quantized `.soen` file and inspect its contents to confirm it matches the expected format for hardware deployment.

In [ ]:
# Inspect the quantized checkpoint via subprocess
_inspect_code = f"""
import sys, json, torch
sys.path.insert(0, {str(SOEN_SRC)!r})

quant_path = {str(Path(audit['quantized_checkpoint']))!r}
obj = torch.load(quant_path, map_location="cpu", weights_only=False)

out = {{"keys": list(obj.keys()), "model_type": obj.get("model_type", "N/A"), "dt_ns": obj.get("dt_ns", "N/A")}}

sd = obj.get("state_dict", {{}})
sd_info = {{}}
for k, v in sd.items():
    if hasattr(v, "shape"):
        sd_info[k] = {{"shape": list(v.shape), "dtype": str(v.dtype)}}
    else:
        sd_info[k] = {{"value": str(v)}}
out["state_dict"] = sd_info

layers = []
for lc in obj.get("layers_config", []):
    layers.append({{"id": lc.get("id"), "type": lc.get("type"), "dim": lc.get("dim"), "desc": lc.get("description", "")}})
out["layers"] = layers

conns = []
for cc in obj.get("connections_config", []):
    conns.append({{"src": cc.get("source_layer_id"), "tgt": cc.get("target_layer_id"),
                  "structure": cc.get("structure", {{}}).get("type", "?"), "learnable": cc.get("learnable")}})
out["connections"] = conns

print(json.dumps(out))
"""

_r = run_soen_python(_inspect_code, check=False)
if _r.returncode == 0:
    info = _json.loads(_r.stdout.strip().split("\n")[-1])

    print(f"=== .soen checkpoint ===")
    print(f"Top-level keys: {info['keys']}")
    print(f"model_type: {info['model_type']}")
    print(f"dt_ns:      {info['dt_ns']}")

    print(f"\n=== state_dict ===")
    for k, v in info["state_dict"].items():
        if "shape" in v:
            print(f"  {k}: shape={v['shape']}, dtype={v['dtype']}")
        else:
            print(f"  {k}: {v['value']}")

    print(f"\n=== Layers ===")
    for l in info["layers"]:
        print(f"  Layer {l['id']}: {l['type']} dim={l['dim']} — {l['desc']}")

    print(f"\n=== Connections ===")
    for c in info["connections"]:
        print(f"  {c['src']}→{c['tgt']}: {c['structure']}, learnable={c['learnable']}")
else:
    print("Checkpoint inspection failed:")
    print(_r.stderr)